In [0]:
# ============================================================
# Notebook : 08_gold_daily_high_low
# Layer    : Gold
# Project  : Real-Time Stock Market Pipeline
#
# Purpose:
# Calculate Daily High and Daily Low
# for every stock.
# ============================================================

In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "realtimedata"

INPUT_TABLE = f"{CATALOG}.gold.ohlc_1min"

OUTPUT_TABLE = f"{CATALOG}.gold.daily_high_low"

In [0]:
ohlc_df = spark.table(INPUT_TABLE)

display(ohlc_df)

symbol,candle_time,open,high,low,close
HDFCBANK,2026-06-29T10:38:00.000Z,798.9,798.9,798.9,798.9
HDFCBANK,2026-06-30T08:40:00.000Z,798.0,798.25,797.75,797.75
HDFCBANK,2026-06-30T08:41:00.000Z,797.6,798.15,797.5,797.75
HDFCBANK,2026-06-30T08:42:00.000Z,797.85,798.25,797.7,798.05
HDFCBANK,2026-06-30T08:43:00.000Z,798.05,798.25,798.0,798.0
HDFCBANK,2026-06-30T09:48:00.000Z,798.0,798.0,797.65,797.65
HDFCBANK,2026-06-30T09:49:00.000Z,797.65,798.2,797.6,798.2
HDFCBANK,2026-06-30T09:50:00.000Z,798.1,798.45,797.95,798.1
HDFCBANK,2026-06-30T09:51:00.000Z,798.1,798.1,797.6,798.0
HDFCBANK,2026-06-30T09:52:00.000Z,798.0,799.0,797.9,798.9


In [0]:
ohlc_df = ohlc_df.withColumn(
    "trading_date",
    F.to_date("candle_time")
)

In [0]:
daily_high_low_df = (
    ohlc_df
    .groupBy(
        "symbol",
        "trading_date"
    )
    .agg(
        F.max("high").alias("day_high"),
        F.min("low").alias("day_low")
    )
)

In [0]:
display(
    daily_high_low_df.orderBy(
        "symbol",
        "trading_date"
    )
)

symbol,trading_date,day_high,day_low
HDFCBANK,2026-06-29,798.9,798.9
HDFCBANK,2026-06-30,800.5,797.5
ICICIBANK,2026-06-29,1387.6,1387.6
ICICIBANK,2026-06-30,1381.4,1367.5
INFY,2026-06-29,1036.7,1036.7
INFY,2026-06-30,1008.8,999.5
RELIANCE,2026-06-29,1301.0,1301.0
RELIANCE,2026-06-30,1294.9,1292.7
TCS,2026-06-29,2097.9,2097.9
TCS,2026-06-30,2035.0,2026.7


In [0]:
(
    daily_high_low_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(OUTPUT_TABLE)
)

In [0]:
display(
    spark.table(OUTPUT_TABLE)
)

symbol,trading_date,day_high,day_low
HDFCBANK,2026-06-30,800.5,797.5
ICICIBANK,2026-06-30,1381.4,1367.5
INFY,2026-06-29,1036.7,1036.7
ICICIBANK,2026-06-29,1387.6,1387.6
TCS,2026-06-29,2097.9,2097.9
INFY,2026-06-30,1008.8,999.5
RELIANCE,2026-06-30,1294.9,1292.7
TCS,2026-06-30,2035.0,2026.7
HDFCBANK,2026-06-29,798.9,798.9
RELIANCE,2026-06-29,1301.0,1301.0
